In [ ]:
import numpy as np
import os
from scipy.interpolate import griddata
from mpl_toolkits.axes_grid1 import make_axes_locatable
import random
import torch
from PINN_magneticfield.train  import PhysicsInformedNN
from PINN_magneticfield.function  import Function


In [ ]:
seed = 1234 
random.seed(seed)          # Python標準乱数
np.random.seed(seed)       # NumPy
torch.manual_seed(seed)    # PyTorch (CPU)

x = np.linspace(0, 63, 64)#規格化(0, 63, 64)でなく(0, 2, 64)など
y = np.linspace(0, 63, 64)
z = np.linspace(0, 63, 64)

x_index = np.arange(64)
y_index = np.arange(64)

max_xyz = np.array([63, 63, 63])
min_xyz = np.array([0.0, 0.0, 0.0])

N_b = 64#境界条件の点
N_f = 64*64*64#コロケーション 

layers = [3, 256, 256, 256, 256,  256,  256, 256,  3]#層の接続
lowlou_f = "/gwork0/yokoyakd/data/b_0.210_0.124.npz"
data = np.load(lowlou_f)
Exa_b = data["b"]#(64, 64, 64, 3)
bottom = Exa_b[:, :, 0, :]

Exa_bx = bottom[:, :, 0:1]
Exa_by = bottom[:, :, 1:2]
Exa_bz = bottom[:, :, 2:3]

X, Y, Z = np.meshgrid(x, y, z)

xyz_star = np.hstack((X.flatten()[:,None], Y.flatten()[:,None], Z.flatten()[:,None]))

bx0 = Exa_bx.flatten()[:, None]#後ろの軸指定の有無
by0 = Exa_by.flatten()[:, None]
bz0 = Exa_bz.flatten()[:, None]

x0 = x
y0 = y

np.random.seed(seed)
xyz_f = np.random.rand(N_f, 3) * 63

In [ ]:
#eval_only
path = "sample/best.pt"
model_dir = os.path.dirname(path)
loss_path = os.path.join(model_dir, "log.txt")
layers, weight1, weight2, TH, lr = Function.load_model_config(loss_path)
model = PhysicsInformedNN(x0, y0, bx0, by0, bz0, xyz_f, layers, min_xyz, max_xyz, TH, weight1, weight2, lr)
model.load_checkpoint(path, optimizer=None)

model.eval()
bx_pred, by_pred, bz_pred= model.predict(xyz_star)

In [ ]:
b = Function.formatting_b(bx_pred, by_pred, bz_pred)
Function.box_liner(b)